# Train a tiny language model from scratch

This notebook trains a small decoder-only Transformer from randomly initialized PyTorch weights. It does not download, call, or fine-tune a pretrained language model. It downloads only the Tiny Shakespeare text corpus and trains a new byte-level BPE tokenizer on the training split.

Hardware fit: the default model has 2 layers, width 128, 4 attention heads, a 128-token context, and a 512-token target vocabulary (about 0.48 million parameters when the vocabulary reaches 512). The inspected computer has 15.8 GiB RAM and a GTX 1650 with 4 GiB VRAM, but its installed PyTorch build is CPU-only, so this run uses CPU. Expect an educational text generator, not a useful general assistant. It may imitate Shakespearean dialogue, repeat itself, and state false things.

The small CC0 conversation file is trained in a separate stage so you can compare its validation loss. It teaches a format only; it does not make the model dependable.

In [1]:
%pip install -r requirements.txt

  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached h5py-3.14.0-cp311-cp311-win_amd64.whl.metadata (2.7 kB)
  Using cached hf_xet-1.7.1-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl (2.9 MB)
Using cached h5py-3.14.0-cp311-cp311-win_amd64.whl (2.9 MB)
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/857.0 kB ? eta -:--:--
   ------------------------------------ --- 786.4/857.0 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 857.0/857.0 kB 1.9 MB/s  0:00:00
Using cached hf_xet-1.7.1-cp38-abi3-win_amd64.whl (3.8 MB)
Using cached httpx2-2.16.0-py3-

In [2]:
from pathlib import Path
import hashlib
import json
import math
import random
import urllib.error
import urllib.request

import torch

try:
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
except ImportError as exc:
    raise ImportError("Install the project dependencies from requirements.txt, then restart the Jupyter kernel.") from exc

ROOT = Path.cwd()
if not (ROOT / "slm.py").is_file():
    raise RuntimeError("Start JupyterLab with the project folder as its working directory.")
import sys
sys.path.insert(0, str(ROOT))
from slm import TinyDecoder, generate

SEED = 17
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device: {DEVICE}")
if DEVICE.type == "cpu":
    print("CPU training selected. No GPU build or CUDA packages will be installed by this notebook.")

PyTorch 2.14.1+cpu; device: cpu
CPU training selected. No GPU build or CUDA packages will be installed by this notebook.


## 1. Prepare the corpus

The source is a 1.06 MB text file containing a subset of Shakespeare's plays. Shakespeare's original works are public domain. The upstream repository describes the file and its source; its MIT software license is not used as a license for the underlying plays. The code records a SHA-256 hash so a run's exact input can be identified.

The downloader fetches text only from the repository's raw GitHub file. If network access is unavailable, download that same file in a browser and save it as data/raw/tinyshakespeare.txt.

In [3]:
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_PATH = RAW_DIR / "tinyshakespeare.txt"
SOURCE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
MAX_BYTES = 3_000_000
RAW_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    try:
        request = urllib.request.Request(SOURCE_URL, headers={"User-Agent": "small-local-language-model-notebook"})
        with urllib.request.urlopen(request, timeout=30) as response:
            content = response.read(MAX_BYTES + 1)
        if len(content) > MAX_BYTES:
            raise ValueError("The downloaded file exceeded the 3 MB safety limit.")
        RAW_PATH.write_bytes(content)
    except (OSError, urllib.error.URLError, TimeoutError, ValueError) as exc:
        raise RuntimeError(
            f"Could not obtain the corpus. Download {SOURCE_URL} in a browser and save it to {RAW_PATH}."
        ) from exc

raw_bytes = RAW_PATH.read_bytes()
if len(raw_bytes) < 100_000 or len(raw_bytes) > MAX_BYTES:
    raise ValueError(f"Corpus size is invalid ({len(raw_bytes):,} bytes); expected 100,000 to {MAX_BYTES:,}.")
try:
    raw_text = raw_bytes.decode("utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError("Corpus must be valid UTF-8 text; replace the local file with the documented source.") from exc
if len(raw_text.strip()) < 100_000:
    raise ValueError("Corpus text is empty or truncated; replace data/raw/tinyshakespeare.txt with the source file.")

corpus_sha256 = hashlib.sha256(raw_bytes).hexdigest()
print(f"Source: {SOURCE_URL}")
print(f"Bytes: {len(raw_bytes):,}; characters: {len(raw_text):,}; SHA-256: {corpus_sha256}")

Source: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Bytes: 1,115,394; characters: 1,115,394; SHA-256: 86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed


## 2. Split before tokenization

Paragraphs are kept intact and divided in source order: the first 90% are training examples and the final 10% are validation examples. Exact duplicate paragraphs are removed before splitting. The tokenizer is fit only on training paragraphs, and token windows are sampled separately inside each split, so no training window crosses into validation.

In [4]:
paragraphs = [part.strip() for part in raw_text.split("\n\n") if part.strip()]
paragraphs = list(dict.fromkeys(paragraphs))
if len(paragraphs) < 20:
    raise ValueError(f"Not enough distinct paragraphs ({len(paragraphs)}); the corpus may be invalid.")

split_at = int(0.90 * len(paragraphs))
train_docs, val_docs = paragraphs[:split_at], paragraphs[split_at:]
if not train_docs or not val_docs or set(train_docs).intersection(val_docs):
    raise ValueError("The corpus split is empty or contains duplicate paragraphs across train and validation.")

print(f"Distinct paragraphs: {len(paragraphs):,}")
print(f"Training paragraphs: {len(train_docs):,}; validation paragraphs: {len(val_docs):,}")

Distinct paragraphs: 7,148
Training paragraphs: 6,433; validation paragraphs: 715


## 3. Train a tokenizer on training text only

Hugging Face Tokenizers is used as a real tokenizer library. This creates a new byte-level BPE vocabulary from train_docs; it loads no pretrained tokenizer or vocabulary.

In [5]:
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
TOKENIZER_PATH = ARTIFACTS / "tokenizer.json"
END_TOKEN = "<|endoftext|>"

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=512,
    min_frequency=2,
    special_tokens=["<unk>", END_TOKEN],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
tokenizer.train_from_iterator(train_docs, trainer=trainer)
tokenizer.save(str(TOKENIZER_PATH))
end_token_id = tokenizer.token_to_id(END_TOKEN)
if end_token_id is None:
    raise RuntimeError("The tokenizer did not create the required end-of-text token.")

def encode_documents(documents):
    ids = []
    for document in documents:
        ids.extend(tokenizer.encode(document).ids)
        ids.append(end_token_id)
    return torch.tensor(ids, dtype=torch.long)

train_tokens = encode_documents(train_docs)
val_tokens = encode_documents(val_docs)
if len(train_tokens) <= 129 or len(val_tokens) <= 129:
    raise ValueError("Train and validation splits each need more than 129 tokens for this context size.")

print(f"Tokenizer vocabulary: {tokenizer.get_vocab_size():,}")
print(f"Training tokens: {len(train_tokens):,}; validation tokens: {len(val_tokens):,}")

Tokenizer vocabulary: 512
Training tokens: 525,429; validation tokens: 43,128


## 4. Define and initialize the decoder

TinyDecoder is implemented in slm.py so the notebook and app use the same architecture. Its token and position embeddings and attention/feed-forward layers are created by fresh PyTorch layer constructors; no model weights are loaded. The output projection is tied to the input token embedding.

In [6]:
MODEL_CONFIG = {
    "vocab_size": tokenizer.get_vocab_size(),
    "block_size": 128,
    "n_layer": 2,
    "n_head": 4,
    "n_embd": 128,
    "dropout": 0.1,
}
model = TinyDecoder(MODEL_CONFIG).to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Randomly initialized parameters: {parameter_count:,}")
print(f"Configuration: {MODEL_CONFIG}")

Randomly initialized parameters: 478,720
Configuration: {'vocab_size': 512, 'block_size': 128, 'n_layer': 2, 'n_head': 4, 'n_embd': 128, 'dropout': 0.1}


## 5. Train with next-token prediction

Each input is a random 128-token training window; its target is the same window shifted one token to the right. Checkpoints are written atomically every evaluation interval. Re-running this cell resumes from the latest compatible checkpoint after an interruption.

In [7]:
MAX_STEPS = 500
EVAL_INTERVAL = 50
EVAL_BATCHES = 12
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
RESUME = True

def get_batch(tokens, batch_size, block_size):
    if len(tokens) <= block_size:
        raise ValueError(f"Need more than {block_size} tokens, found {len(tokens)}.")
    starts = torch.randint(len(tokens) - block_size, (batch_size,))
    x = torch.stack([tokens[i:i + block_size] for i in starts]).to(DEVICE)
    y = torch.stack([tokens[i + 1:i + block_size + 1] for i in starts]).to(DEVICE)
    return x, y

@torch.no_grad()
def estimate_loss(current_model, train_stream, val_stream, block_size=128):
    current_model.eval()
    values = {}
    for name, stream in (("train", train_stream), ("validation", val_stream)):
        losses = []
        for _ in range(EVAL_BATCHES):
            x, y = get_batch(stream, BATCH_SIZE, block_size)
            _, loss = current_model(x, y)
            losses.append(loss.item())
        values[name] = sum(losses) / len(losses)
    current_model.train()
    return values

def atomic_torch_save(value, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)

def atomic_json_save(value, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temporary.replace(path)

def train_stage(current_model, train_stream, val_stream, *, stage, signature,
                max_steps, eval_interval, batch_size, block_size, learning_rate,
                initial_state=None):
    last_path = ARTIFACTS / f"{stage}_last.pt"
    best_path = ARTIFACTS / f"{stage}_best.pt"
    history_path = ARTIFACTS / f"{stage}_loss_history.json"
    optimizer = torch.optim.AdamW(current_model.parameters(), lr=learning_rate)
    start_step, best_val = 0, float("inf")
    history = []

    if RESUME and last_path.exists():
        try:
            saved = torch.load(last_path, map_location=DEVICE, weights_only=True)
            if saved["signature"] != signature:
                raise RuntimeError("The saved checkpoint belongs to different data or settings. Move the artifacts aside and start fresh.")
            current_model.load_state_dict(saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            start_step = saved["step"] + 1
            best_val = saved["best_val"]
            if history_path.exists():
                history = json.loads(history_path.read_text(encoding="utf-8"))
            print(f"Resuming {stage} at step {start_step}.")
        except Exception as exc:
            raise RuntimeError(f"Could not resume {stage}: {exc}") from exc
    elif initial_state is not None:
        current_model.load_state_dict(initial_state)

    for step in range(start_step, max_steps):
        current_model.train()
        x, y = get_batch(train_stream, batch_size, block_size)
        _, loss = current_model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(current_model.parameters(), 1.0)
        optimizer.step()

        if (step + 1) % eval_interval == 0 or step + 1 == max_steps:
            measured = estimate_loss(current_model, train_stream, val_stream, block_size)
            row = {"stage": stage, "step": step + 1, **measured}
            history.append(row)
            atomic_json_save(history, history_path)
            if measured["validation"] < best_val:
                best_val = measured["validation"]
                atomic_torch_save(current_model.state_dict(), best_path)
            atomic_torch_save({
                "signature": signature,
                "step": step,
                "best_val": best_val,
                "model": current_model.state_dict(),
                "optimizer": optimizer.state_dict(),
            }, last_path)
            print(
                f'{stage} {step + 1:>4}/{max_steps}: '
                f'train loss {measured["train"]:.3f}, '
                f'validation loss {measured["validation"]:.3f}, '
                f'validation perplexity {math.exp(min(measured["validation"], 20)):.2f}'
            )

    if not best_path.exists():
        raise RuntimeError(f"{stage} did not produce a checkpoint; check the training log and available memory.")
    current_model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    return current_model, history

pretrain_signature = hashlib.sha256(
    f"{corpus_sha256}:{MODEL_CONFIG}:{MAX_STEPS}:{LEARNING_RATE}".encode()
).hexdigest()
model, pretrain_history = train_stage(
    model, train_tokens, val_tokens,
    stage="pretrain",
    signature=pretrain_signature,
    max_steps=MAX_STEPS,
    eval_interval=EVAL_INTERVAL,
    batch_size=BATCH_SIZE,
    block_size=MODEL_CONFIG["block_size"],
    learning_rate=LEARNING_RATE,
)

Resuming pretrain at step 500.


## 6. Evaluate pretraining and inspect a sample

Perplexity is exp(validation loss). It measures next-token prediction on this held-out text, not factual accuracy or usefulness. The sample below is generated by the newly trained checkpoint and may be incoherent.

In [8]:
pretrain_metrics = estimate_loss(model, train_tokens, val_tokens)
print(f"Pretraining validation loss: {pretrain_metrics['validation']:.3f}")
print(f"Pretraining validation perplexity: {math.exp(min(pretrain_metrics['validation'], 20)):.2f}")
print("\nGenerated sample (unreliable; based on this narrow literary corpus):")
print(generate(
    model, tokenizer,
    "The traveler looked toward the quiet road.",
    end_token_id,
    max_new_tokens=80,
    temperature=0.9,
))

Pretraining validation loss: 4.669
Pretraining validation perplexity: 106.62

Generated sample (unreliable; based on this narrow literary corpus):



## 7. Instruction tuning on openly licensed examples

The separate examples in data/instructions_cc0.jsonl were written for this project and dedicated to CC0 1.0. This stage holds out complete conversations and continues from the best pretraining checkpoint. The dataset is deliberately small: it can teach the User/Assistant pattern but cannot provide broad assistant ability.

In [9]:
instruction_path = DATA_DIR / "instructions_cc0.jsonl"
if not instruction_path.is_file():
    raise FileNotFoundError(f"Missing instruction examples: {instruction_path}")

try:
    conversations = [json.loads(line) for line in instruction_path.read_text(encoding="utf-8").splitlines() if line.strip()]
except json.JSONDecodeError as exc:
    raise ValueError(f"Invalid JSON in {instruction_path}: {exc}") from exc
if len(conversations) < 10 or any(
    not isinstance(item, dict)
    or not isinstance(item.get("user"), str) or not item["user"].strip()
    or not isinstance(item.get("assistant"), str) or not item["assistant"].strip()
    for item in conversations
):
    raise ValueError("Instruction data needs at least 10 objects with non-empty user and assistant strings.")

shuffled = conversations.copy()
random.Random(SEED).shuffle(shuffled)
instruction_cut = max(1, min(len(shuffled) - 1, int(0.85 * len(shuffled))))
train_conversations, val_conversations = shuffled[:instruction_cut], shuffled[instruction_cut:]

def encode_conversations(items):
    stream = []
    for item in items:
        turn = f"User: {item['user']}\nAssistant: {item['assistant']}"
        stream.extend(tokenizer.encode(turn).ids)
        stream.append(end_token_id)
    return torch.tensor(stream, dtype=torch.long)

instruction_train = encode_conversations(train_conversations)
instruction_val = encode_conversations(val_conversations)
instruction_context = min(64, MODEL_CONFIG["block_size"])
if len(instruction_train) <= instruction_context or len(instruction_val) <= instruction_context:
    raise ValueError("Instruction train and validation splits need more than 64 tokens each.")

instruction_model = TinyDecoder(MODEL_CONFIG).to(DEVICE)
pretrain_best = torch.load(ARTIFACTS / "pretrain_best.pt", map_location=DEVICE, weights_only=True)
instruction_model.load_state_dict(pretrain_best)
instruction_signature = hashlib.sha256(
    f"{hashlib.sha256(instruction_path.read_bytes()).hexdigest()}:{corpus_sha256}:{MODEL_CONFIG}".encode()
).hexdigest()
instruction_model, instruction_history = train_stage(
    instruction_model, instruction_train, instruction_val,
    stage="instruction",
    signature=instruction_signature,
    max_steps=80,
    eval_interval=10,
    batch_size=8,
    block_size=instruction_context,
    learning_rate=1e-4,
    initial_state=pretrain_best,
)

instruction_metrics = estimate_loss(
    instruction_model, instruction_train, instruction_val, instruction_context
)
print(f"Instruction validation loss: {instruction_metrics['validation']:.3f}")
print(f"Instruction validation perplexity: {math.exp(min(instruction_metrics['validation'], 20)):.2f}")

Resuming instruction at step 80.
Instruction validation loss: 4.301
Instruction validation perplexity: 73.79


## 8. Export the local model

The app uses only these small local artifacts: model.pt, config.json, and tokenizer.json. Checkpoints and loss histories stay under the ignored artifacts folder and should not be committed or published without a separate review.

In [10]:
final_model = instruction_model
final_history = {
    "pretraining": pretrain_history,
    "instruction_tuning": instruction_history,
}
final_config = {
    **MODEL_CONFIG,
    "parameter_count": parameter_count,
    "end_token_id": end_token_id,
    "corpus_sha256": corpus_sha256,
    "instruction_tuned": True,
    "limitations": "Educational Shakespeare-corpus model; not a general or reliable assistant.",
}
atomic_torch_save(final_model.state_dict(), ARTIFACTS / "model.pt")
atomic_json_save(final_config, ARTIFACTS / "config.json")
atomic_json_save(final_history, ARTIFACTS / "loss_history.json")

print(f"Exported model to {ARTIFACTS / 'model.pt'}")
print(f"Validation loss: {instruction_metrics['validation']:.3f}")
print(f"Validation perplexity: {math.exp(min(instruction_metrics['validation'], 20)):.2f}")
print("\nGenerated sample (not a verified factual answer):")
print(generate(
    final_model, tokenizer,
    "User: What does a tokenizer do?\nAssistant:",
    end_token_id,
    max_new_tokens=48,
    temperature=0.7,
))

Exported model to C:\Users\shree\OneDrive\Documents\Github\OnGoing\SLM\artifacts\model.pt
Validation loss: 4.301
Validation perplexity: 73.79

Generated sample (not a verified factual answer):
 S As once.
